# EXP029: gamma別の方策と項目パラメータ比較

保存済みの通常DQNモデルへ共通の推定能力値 $\hat\theta$ を入力し、各モデル内でQ値が高い項目・低い項目の3PLパラメータ $(a,b,c)$ を比較する。さらに、同じ推定能力値におけるMFIのFisher情報量上位・下位項目を基準として重ねる。

- 対象: `bank 1`, `normal prior`, 通常DQN
- gamma: `0, 0.1, 0.3, 0.5, 0.9, 1`
- theta: `-3, -2, -1, 0, 1, 2, 3`
- 上位・下位: 各10項目
- 状態: 未出題項目マスクを適用しない初回選択時の状態

テスト時の方策はgreedyであるため、固定したthetaでは最大Q値の1項目が実際に選択される。ここでは「選ばれやすい項目」をQ値上位、「選ばれにくい項目」をQ値下位と定義する。Q値の絶対的な尺度はモデルごとに異なるため、gamma間ではQ値そのものではなくモデル内順位と項目パラメータを比較する。

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import display
from torch import nn

pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda value: f"{value:.4f}")

In [ ]:
ROOT = next(
    path
    for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "pyproject.toml").exists()
)
EXP_DIR = ROOT / "EXP029"
MODEL_DIR = EXP_DIR / "models"
RESULTS_DIR = EXP_DIR / "results"

BANK_TYPE = "uncor"
BANK_ID = 1
PRIOR = "normal"
GAMMA_LABELS = ["0", "0.1", "0.3", "0.5", "0.9", "1"]
THETA_VALUES = np.array([-3, -2, -1, 0, 1, 2, 3], dtype=np.float32)
TOP_K = 10

FIRST_HIDDEN = 50
SECOND_HIDDEN = 30
DROPOUT_RATE = 0.0

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
RANKING_PATH = (
    RESULTS_DIR / f"policy_q_rankings_{BANK_TYPE}_{BANK_ID}_DQN_MLE_{PRIOR}.csv"
)
SUMMARY_PATH = (
    RESULTS_DIR
    / f"policy_item_parameter_summary_{BANK_TYPE}_{BANK_ID}_DQN_MLE_{PRIOR}.csv"
)
MFI_RANKING_PATH = RESULTS_DIR / f"mfi_rankings_{BANK_TYPE}_{BANK_ID}_MLE.csv"
MFI_SUMMARY_PATH = (
    RESULTS_DIR / f"mfi_item_parameter_summary_{BANK_TYPE}_{BANK_ID}_MLE.csv"
)
FIGURE_PATH = (
    RESULTS_DIR
    / f"policy_item_parameters_with_MFI_{BANK_TYPE}_{BANK_ID}_DQN_MLE_{PRIOR}.png"
)

print(f"Repository root: {ROOT}")
print(f"theta values: {THETA_VALUES.tolist()}")
print(f"top/bottom k: {TOP_K}")

In [ ]:
class Net(nn.Module):
    def __init__(
        self,
        input_size: int,
        first_hidden: int,
        second_hidden: int,
        action_space: int,
        dropout_rate: float,
    ) -> None:
        super().__init__()
        self.fc1 = nn.Linear(input_size, first_hidden)
        self.fc2 = nn.Linear(first_hidden, second_hidden)
        self.out = nn.Linear(second_hidden, action_space)
        self.dropout = nn.Dropout(dropout_rate)

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        hidden = F.relu(self.dropout(self.fc1(inputs)))
        hidden = F.relu(self.dropout(self.fc2(hidden)))
        return self.out(hidden)

In [ ]:
item_bank_path = (
    ROOT / "data" / "uncorrelated_banks" / f"item_bank_{BANK_TYPE}_{BANK_ID}.csv"
)
item_bank = pd.read_csv(item_bank_path)[["a", "b", "c"]].copy()
item_bank.index = np.arange(1, len(item_bank) + 1)
item_bank.index.name = "itemID"
action_space = len(item_bank)
item_bank_array = item_bank[["a", "b", "c"]].to_numpy()

models: dict[str, Net] = {}
for gamma in GAMMA_LABELS:
    model_path = MODEL_DIR / f"dqn_MLE_{PRIOR}_{BANK_TYPE}_{BANK_ID}_gamma_{gamma}.pt"
    if not model_path.exists():
        raise FileNotFoundError(f"Model not found: {model_path}")

    model = Net(
        input_size=1,
        first_hidden=FIRST_HIDDEN,
        second_hidden=SECOND_HIDDEN,
        action_space=action_space,
        dropout_rate=DROPOUT_RATE,
    )
    state_dict = torch.load(model_path, map_location="cpu", weights_only=True)
    model.load_state_dict(state_dict)
    model.eval()
    models[gamma] = model

print(f"Item bank: {item_bank_path} ({action_space} items)")
print(f"Loaded {len(models)} models: {list(models)}")

## Q値上位・下位項目

各gamma・thetaの組合せについて、Q値をモデル内で降順または昇順に並べる。同値の場合はDQNの `argmax` と同じくitemIDの小さい項目を先にする。

In [ ]:
def ranked_items(
    q_values: np.ndarray,
    gamma: str,
    theta: float,
    group: str,
    item_indices: np.ndarray,
) -> pd.DataFrame:
    selected = item_bank.iloc[item_indices].reset_index()
    selected.insert(0, "rank", np.arange(1, len(item_indices) + 1))
    selected.insert(0, "rank_group", group)
    selected.insert(0, "theta", theta)
    selected.insert(0, "gamma", gamma)
    selected["Q_value"] = q_values[item_indices]
    return selected


theta_tensor = torch.from_numpy(THETA_VALUES).reshape(-1, 1)
ranking_frames: list[pd.DataFrame] = []
q_values_by_gamma: dict[str, np.ndarray] = {}

for gamma, model in models.items():
    with torch.no_grad():
        q_matrix = model(theta_tensor).cpu().numpy()
    q_values_by_gamma[gamma] = q_matrix

    for theta, q_values in zip(THETA_VALUES, q_matrix, strict=True):
        top_indices = np.argsort(-q_values, kind="stable")[:TOP_K]
        bottom_indices = np.argsort(q_values, kind="stable")[:TOP_K]
        ranking_frames.extend(
            [
                ranked_items(q_values, gamma, float(theta), "top", top_indices),
                ranked_items(q_values, gamma, float(theta), "bottom", bottom_indices),
            ]
        )

rankings = pd.concat(ranking_frames, ignore_index=True)
rankings.to_csv(RANKING_PATH, index=False)
print(f"Saved rankings: {RANKING_PATH}")
display(rankings.head(20))

In [ ]:
parameter_summary = (
    rankings.groupby(["gamma", "theta", "rank_group"], sort=False)
    .agg(
        n_items=("itemID", "size"),
        a_mean=("a", "mean"),
        a_std=("a", "std"),
        b_mean=("b", "mean"),
        b_std=("b", "std"),
        c_mean=("c", "mean"),
        c_std=("c", "std"),
        q_mean=("Q_value", "mean"),
    )
    .reset_index()
)
parameter_summary.to_csv(SUMMARY_PATH, index=False)
print(f"Saved parameter summary: {SUMMARY_PATH}")
display(parameter_summary)

## MFIのFisher情報量上位・下位項目

EXP027のMFIと同じ3PL Fisher情報量を使用し、各thetaで情報量上位・下位10項目を求める。未出題マスクなしの初回選択では、情報量1位の項目がMFIによって選択される。

In [ ]:
def fisher_information(
    item_parameters: np.ndarray, theta: float, scaling: float = 1.0
) -> np.ndarray:
    a = item_parameters[:, 0]
    b = item_parameters[:, 1]
    c = item_parameters[:, 2]
    return (
        scaling**2
        * a**2
        * (1 - c)
        / (c + np.exp(scaling * a * (theta - b)))
        / (1 + np.exp(-scaling * a * (theta - b))) ** 2
    )


def ranked_mfi_items(
    information: np.ndarray,
    theta: float,
    group: str,
    item_indices: np.ndarray,
) -> pd.DataFrame:
    selected = item_bank.iloc[item_indices].reset_index()
    selected.insert(0, "rank", np.arange(1, len(item_indices) + 1))
    selected.insert(0, "rank_group", group)
    selected.insert(0, "theta", theta)
    selected["FI_value"] = information[item_indices]
    return selected


mfi_frames: list[pd.DataFrame] = []
for theta in THETA_VALUES:
    information = fisher_information(item_bank_array, float(theta))
    top_indices = np.argsort(-information, kind="stable")[:TOP_K]
    bottom_indices = np.argsort(information, kind="stable")[:TOP_K]
    mfi_frames.extend(
        [
            ranked_mfi_items(information, float(theta), "top", top_indices),
            ranked_mfi_items(information, float(theta), "bottom", bottom_indices),
        ]
    )

mfi_rankings = pd.concat(mfi_frames, ignore_index=True)
mfi_rankings.to_csv(MFI_RANKING_PATH, index=False)
mfi_parameter_summary = (
    mfi_rankings.groupby(["theta", "rank_group"], sort=False)
    .agg(
        n_items=("itemID", "size"),
        a_mean=("a", "mean"),
        a_std=("a", "std"),
        b_mean=("b", "mean"),
        b_std=("b", "std"),
        c_mean=("c", "mean"),
        c_std=("c", "std"),
        fi_mean=("FI_value", "mean"),
    )
    .reset_index()
)
mfi_parameter_summary.to_csv(MFI_SUMMARY_PATH, index=False)
print(f"Saved MFI rankings: {MFI_RANKING_PATH}")
print(f"Saved MFI parameter summary: {MFI_SUMMARY_PATH}")
display(mfi_parameter_summary)

## thetaごとの詳細表

以下のセルではthetaを1つ指定し、全gammaの上位・下位項目を確認できる。

In [ ]:
THETA_TO_DISPLAY = 0.0

detail = rankings.loc[rankings["theta"] == THETA_TO_DISPLAY].copy()
if detail.empty:
    raise ValueError(
        f"THETA_TO_DISPLAY={THETA_TO_DISPLAY} is not in {THETA_VALUES.tolist()}"
    )
display(detail.set_index(["gamma", "rank_group", "rank"]))
display(
    mfi_rankings.loc[mfi_rankings["theta"] == THETA_TO_DISPLAY].set_index(
        ["rank_group", "rank"]
    )
)

## 実際のgreedy選択（Q値1位）

未出題マスクなしの場合に実際に選ばれる1位項目だけを一覧にする。

In [ ]:
greedy_items = rankings.loc[
    (rankings["rank_group"] == "top") & (rankings["rank"] == 1)
].copy()
display(
    greedy_items.pivot(index="gamma", columns="theta", values=["itemID", "a", "b", "c"])
)
mfi_greedy_items = mfi_rankings.loc[
    (mfi_rankings["rank_group"] == "top") & (mfi_rankings["rank"] == 1)
]
display(mfi_greedy_items.set_index("theta"))

## 上位・下位10項目の平均パラメータ

青実線はDQNのQ値上位10項目、オレンジ破線は下位10項目の平均を表す。緑点線と赤一点鎖線は、それぞれ同じthetaにおけるMFIのFisher情報量上位・下位10項目の平均であり、gammaに依存しない基準線として示す。縦軸の範囲はパラメータ・thetaごとに独立している。

In [ ]:
gamma_positions = np.arange(len(GAMMA_LABELS))
parameters = [
    ("a_mean", "Discrimination a"),
    ("b_mean", "Difficulty b"),
    ("c_mean", "Guessing c"),
]
fig, axes = plt.subplots(
    len(parameters),
    len(THETA_VALUES),
    figsize=(22, 9),
    sharex=True,
    constrained_layout=True,
)

for row, (column, label) in enumerate(parameters):
    for col, theta in enumerate(THETA_VALUES):
        axis = axes[row, col]
        subset = parameter_summary.loc[parameter_summary["theta"] == theta]
        for group, linestyle, marker, group_label in [
            ("top", "-", "o", "DQN top"),
            ("bottom", "--", "s", "DQN bottom"),
        ]:
            values = (
                subset.loc[subset["rank_group"] == group]
                .set_index("gamma")
                .loc[GAMMA_LABELS, column]
            )
            axis.plot(
                gamma_positions,
                values,
                linestyle=linestyle,
                marker=marker,
                linewidth=1.5,
                markersize=4,
                label=group_label,
            )
        mfi_values = mfi_parameter_summary.loc[
            mfi_parameter_summary["theta"] == theta
        ].set_index("rank_group")
        axis.axhline(
            mfi_values.loc["top", column],
            color="tab:green",
            linestyle=":",
            linewidth=2.0,
            label="MFI top",
        )
        axis.axhline(
            mfi_values.loc["bottom", column],
            color="tab:red",
            linestyle="-.",
            linewidth=1.8,
            label="MFI bottom",
        )
        if row == 0:
            axis.set_title(rf"$\hat{{\theta}}={theta:g}$")
        if col == 0:
            axis.set_ylabel(label)
        if row == len(parameters) - 1:
            axis.set_xticks(gamma_positions, GAMMA_LABELS)
            axis.set_xlabel("gamma")
        axis.grid(alpha=0.25)

handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", bbox_to_anchor=(0.5, 1.03), ncol=4)
fig.suptitle("EXP029: DQN Q-value ranking vs MFI information ranking", y=1.065)
fig.savefig(FIGURE_PATH, dpi=180, bbox_inches="tight")
plt.show()
print(f"Saved figure: {FIGURE_PATH}")

# Bank 2での再分析

bank 2で保存されている通常DQNモデルを、bank 1と同じ条件で分析する。bank 2にはgamma=1のモデルがないため、利用可能な `0, 0.1, 0.3, 0.5, 0.7, 0.9` を対象とする。

In [ ]:
BANK2_ID = 2
BANK2_GAMMA_LABELS = ["0", "0.1", "0.3", "0.5", "0.7", "0.9"]

bank2_item_bank_path = (
    ROOT / "data" / "uncorrelated_banks" / f"item_bank_{BANK_TYPE}_{BANK2_ID}.csv"
)
bank2_item_bank = pd.read_csv(bank2_item_bank_path)[["a", "b", "c"]].copy()
bank2_item_bank.index = np.arange(1, len(bank2_item_bank) + 1)
bank2_item_bank.index.name = "itemID"
bank2_item_bank_array = bank2_item_bank[["a", "b", "c"]].to_numpy()
bank2_action_space = len(bank2_item_bank)

bank2_models: dict[str, Net] = {}
for gamma in BANK2_GAMMA_LABELS:
    model_path = MODEL_DIR / f"dqn_MLE_{PRIOR}_{BANK_TYPE}_{BANK2_ID}_gamma_{gamma}.pt"
    if not model_path.exists():
        raise FileNotFoundError(f"Model not found: {model_path}")
    model = Net(
        input_size=1,
        first_hidden=FIRST_HIDDEN,
        second_hidden=SECOND_HIDDEN,
        action_space=bank2_action_space,
        dropout_rate=DROPOUT_RATE,
    )
    model.load_state_dict(torch.load(model_path, map_location="cpu", weights_only=True))
    model.eval()
    bank2_models[gamma] = model


def bank2_ranked_items(
    scores: np.ndarray,
    theta: float,
    group: str,
    item_indices: np.ndarray,
    score_column: str,
    gamma: str | None = None,
) -> pd.DataFrame:
    selected = bank2_item_bank.iloc[item_indices].reset_index()
    selected.insert(0, "rank", np.arange(1, len(item_indices) + 1))
    selected.insert(0, "rank_group", group)
    selected.insert(0, "theta", theta)
    if gamma is not None:
        selected.insert(0, "gamma", gamma)
    selected[score_column] = scores[item_indices]
    return selected


bank2_dqn_frames: list[pd.DataFrame] = []
for gamma, model in bank2_models.items():
    with torch.no_grad():
        q_matrix = model(theta_tensor).cpu().numpy()
    for theta, q_values in zip(THETA_VALUES, q_matrix, strict=True):
        top_indices = np.argsort(-q_values, kind="stable")[:TOP_K]
        bottom_indices = np.argsort(q_values, kind="stable")[:TOP_K]
        bank2_dqn_frames.extend(
            [
                bank2_ranked_items(
                    q_values,
                    float(theta),
                    "top",
                    top_indices,
                    "Q_value",
                    gamma,
                ),
                bank2_ranked_items(
                    q_values,
                    float(theta),
                    "bottom",
                    bottom_indices,
                    "Q_value",
                    gamma,
                ),
            ]
        )

bank2_rankings = pd.concat(bank2_dqn_frames, ignore_index=True)
bank2_parameter_summary = (
    bank2_rankings.groupby(["gamma", "theta", "rank_group"], sort=False)
    .agg(
        n_items=("itemID", "size"),
        a_mean=("a", "mean"),
        a_std=("a", "std"),
        b_mean=("b", "mean"),
        b_std=("b", "std"),
        c_mean=("c", "mean"),
        c_std=("c", "std"),
        q_mean=("Q_value", "mean"),
    )
    .reset_index()
)

bank2_mfi_frames: list[pd.DataFrame] = []
for theta in THETA_VALUES:
    information = fisher_information(bank2_item_bank_array, float(theta))
    top_indices = np.argsort(-information, kind="stable")[:TOP_K]
    bottom_indices = np.argsort(information, kind="stable")[:TOP_K]
    bank2_mfi_frames.extend(
        [
            bank2_ranked_items(
                information, float(theta), "top", top_indices, "FI_value"
            ),
            bank2_ranked_items(
                information,
                float(theta),
                "bottom",
                bottom_indices,
                "FI_value",
            ),
        ]
    )

bank2_mfi_rankings = pd.concat(bank2_mfi_frames, ignore_index=True)
bank2_mfi_parameter_summary = (
    bank2_mfi_rankings.groupby(["theta", "rank_group"], sort=False)
    .agg(
        n_items=("itemID", "size"),
        a_mean=("a", "mean"),
        a_std=("a", "std"),
        b_mean=("b", "mean"),
        b_std=("b", "std"),
        c_mean=("c", "mean"),
        c_std=("c", "std"),
        fi_mean=("FI_value", "mean"),
    )
    .reset_index()
)

bank2_ranking_path = (
    RESULTS_DIR / f"policy_q_rankings_{BANK_TYPE}_{BANK2_ID}_DQN_MLE_{PRIOR}.csv"
)
bank2_summary_path = (
    RESULTS_DIR
    / f"policy_item_parameter_summary_{BANK_TYPE}_{BANK2_ID}_DQN_MLE_{PRIOR}.csv"
)
bank2_mfi_ranking_path = RESULTS_DIR / f"mfi_rankings_{BANK_TYPE}_{BANK2_ID}_MLE.csv"
bank2_mfi_summary_path = (
    RESULTS_DIR / f"mfi_item_parameter_summary_{BANK_TYPE}_{BANK2_ID}_MLE.csv"
)
bank2_figure_path = (
    RESULTS_DIR
    / f"policy_item_parameters_with_MFI_{BANK_TYPE}_{BANK2_ID}_DQN_MLE_{PRIOR}.png"
)
bank2_rankings.to_csv(bank2_ranking_path, index=False)
bank2_parameter_summary.to_csv(bank2_summary_path, index=False)
bank2_mfi_rankings.to_csv(bank2_mfi_ranking_path, index=False)
bank2_mfi_parameter_summary.to_csv(bank2_mfi_summary_path, index=False)

gamma_positions = np.arange(len(BANK2_GAMMA_LABELS))
fig, axes = plt.subplots(
    len(parameters),
    len(THETA_VALUES),
    figsize=(22, 9),
    sharex=True,
    constrained_layout=True,
)
for row, (column, label) in enumerate(parameters):
    for col, theta in enumerate(THETA_VALUES):
        axis = axes[row, col]
        subset = bank2_parameter_summary.loc[bank2_parameter_summary["theta"] == theta]
        for group, linestyle, marker, group_label in [
            ("top", "-", "o", "DQN top"),
            ("bottom", "--", "s", "DQN bottom"),
        ]:
            values = (
                subset.loc[subset["rank_group"] == group]
                .set_index("gamma")
                .loc[BANK2_GAMMA_LABELS, column]
            )
            axis.plot(
                gamma_positions,
                values,
                linestyle=linestyle,
                marker=marker,
                linewidth=1.5,
                markersize=4,
                label=group_label,
            )
        mfi_values = bank2_mfi_parameter_summary.loc[
            bank2_mfi_parameter_summary["theta"] == theta
        ].set_index("rank_group")
        axis.axhline(
            mfi_values.loc["top", column],
            color="tab:green",
            linestyle=":",
            linewidth=2.0,
            label="MFI top",
        )
        axis.axhline(
            mfi_values.loc["bottom", column],
            color="tab:red",
            linestyle="-.",
            linewidth=1.8,
            label="MFI bottom",
        )
        if row == 0:
            axis.set_title(rf"$\hat{{\theta}}={theta:g}$")
        if col == 0:
            axis.set_ylabel(label)
        if row == len(parameters) - 1:
            axis.set_xticks(gamma_positions, BANK2_GAMMA_LABELS)
            axis.set_xlabel("gamma")
        axis.grid(alpha=0.25)

handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", bbox_to_anchor=(0.5, 1.03), ncol=4)
fig.suptitle("EXP029 bank 2: DQN Q-value ranking vs MFI information ranking", y=1.065)
fig.savefig(bank2_figure_path, dpi=180, bbox_inches="tight")
plt.show()

bank2_greedy = bank2_rankings.loc[
    (bank2_rankings["rank_group"] == "top") & (bank2_rankings["rank"] == 1)
]
display(
    bank2_greedy.pivot(index="gamma", columns="theta", values=["itemID", "a", "b", "c"])
)
print(f"Saved bank 2 DQN rankings: {bank2_ranking_path}")
print(f"Saved bank 2 DQN summary: {bank2_summary_path}")
print(f"Saved bank 2 MFI rankings: {bank2_mfi_ranking_path}")
print(f"Saved bank 2 MFI summary: {bank2_mfi_summary_path}")
print(f"Saved bank 2 figure: {bank2_figure_path}")

## 解釈上の注意

- この分析は現在の推定値 $\hat\theta$ を状態としてモデルへ入力する。真のthetaを直接入力しているわけではない。
- 初回選択を比較するため出題済み項目マスクを使わない。2問目以降は、それまでに選択された項目を除いた残存項目内で順位が変わる。
- greedy方策では固定状態における選択確率は定義されず、Q値1位の項目が決定的に選択される。上位10項目は選択候補の順位を見るための補助指標である。
- MFIも固定状態ではFisher情報量1位の項目を決定的に選択する。MFIの上位・下位10項目は、DQNと同じ件数で項目パラメータを比較するための補助指標である。
- Q値の絶対値はgammaの異なるモデル間で直接比較しない。
- 各gammaは現状1学習runであり、学習乱数が固定されていないため、観察された差をgammaだけの効果とは断定できない。